

The continuous SIR model is:

$$
\frac{dS}{dt} = -\beta SI
$$

$$
\frac{dI}{dt} = \beta SI - \gamma I
$$

$$
\frac{dR}{dt} = \gamma I
$$

where:

$$
S(t) = \text{susceptible population}
$$

$$
I(t) = \text{infectious population}
$$

$$
R(t) = \text{recovered population}
$$

$$
\beta = \text{transmission rate}
$$

$$
\gamma = \text{recovery rate}
$$

---




In [4]:
#!pip install ipympl ipywidgets
!pip install scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 96.0 MB/s  0:00:00m eta 0:00:01

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


# SIR MODEL


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from scipy.integrate import solve_ivp
from IPython.display import display, clear_output

# -----------------------------
# Initial model parameters
# -----------------------------
initial_transmission_rate = 0.0025
initial_recovery_days = 7
vaccination_speed = 0.001  # Assuming a constant vaccination rate

S0 = 250
I0 = 1
R0 = 0

days = 160
dt = 0.1
time = np.arange(0, days, dt)


# -----------------------------
# SIR simulation function
# -----------------------------
def simulate_sir(transmission_rate, recovery_days):
    recovery_rate = 1 / recovery_days

    def sir_system(t, state):
        S, I, R = state
        dS_dt = -transmission_rate * S * I
        dI_dt = transmission_rate * S * I - recovery_rate * I
        dR_dt = recovery_rate * I
        return [dS_dt, dI_dt, dR_dt]

    solution = solve_ivp(
        sir_system,
        (time[0], time[-1]),
        [S0, I0, R0],
        t_eval=time,
    )

    if not solution.success:
        raise RuntimeError(f"SIR integration failed: {solution.message}")

    return solution.y


# -----------------------------
# Widgets
# -----------------------------
beta_slider = widgets.FloatSlider(
    value=initial_transmission_rate,
    min=0.0001,
    max=0.01,
    step=0.0001,
    description="β:",
    readout_format=".4f"
)

recovery_slider = widgets.IntSlider(
    value=initial_recovery_days,
    min=1,
    max=30,
    step=1,
    description="Recovery days:"
)

reset_button = widgets.Button(
    description="Reset",
    button_style="warning"
)

output = widgets.Output()


# -----------------------------
# Plot function
# -----------------------------
def draw_plot():
    transmission_rate = beta_slider.value
    recovery_days = recovery_slider.value

    S, I, R = simulate_sir(transmission_rate, recovery_days)

    with output:
        clear_output(wait=True)

        plt.figure(figsize=(10, 6))
        plt.plot(time, S, label="Susceptible S(t)")
        plt.plot(time, I, label="Infectious I(t)")
        plt.plot(time, R, label="Recovered R(t)")

        plt.title(
            f"SIR Model: β = {transmission_rate:.4f}, "
            f"recovery time = {recovery_days} days"
        )
        plt.xlabel("Time in days")
        plt.ylabel("Number of people")
        plt.ylim(0, max(S.max(), I.max(), R.max()) * 1.1)
        plt.legend()
        plt.grid(True)
        plt.show()


# -----------------------------
# Update function
# -----------------------------
def update_plot(change=None):
    draw_plot()


# -----------------------------
# Reset function
# -----------------------------
def reset_model(button):
    beta_slider.value = initial_transmission_rate
    recovery_slider.value = initial_recovery_days

    draw_plot()


beta_slider.observe(update_plot, names="value")
recovery_slider.observe(update_plot, names="value")

reset_button.on_click(reset_model)


display(beta_slider, recovery_slider, reset_button, output)

draw_plot()

FloatSlider(value=0.0025, description='β:', max=0.01, min=0.0001, readout_format='.4f', step=0.0001)

IntSlider(value=7, description='Recovery days:', max=30, min=1)

Button(button_style='warning', description='Reset', style=ButtonStyle())

Output()

# SIR WITH VACCINATION STATUS



The continuous SIR model is:

$$
\frac{dS}{dt} = -\beta SI -\delta I
$$

$$
\frac{dI}{dt} = \beta SI - \gamma I
$$

$$
\frac{dR}{dt} = \gamma I
$$

$$
\frac{dV}{dt} = \delta I
$$

where:

$$
S(t) = \text{susceptible population}
$$

$$
I(t) = \text{infectious population}
$$

$$
R(t) = \text{recovered population}
$$

$$
V(t) = \text{immunized population}
$$

$$
\beta = \text{transmission rate}
$$

$$
\gamma = \text{recovery rate}
$$

$$
\delta = \text{immunization rate}
$$
---




In [23]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from scipy.integrate import solve_ivp
from IPython.display import display, clear_output

# -----------------------------
# Initial model parameters
# -----------------------------
initial_transmission_rate = 0.0025
initial_recovery_days = 7 #7 days for recovery
initial_immunization_speed = 0.01 # Assuming a constant vaccination rate of 1 per day

S0 = 250
I0 = 1
R0 = 0
V0 = 0

days = 160
dt = 0.1
time = np.arange(0, days, dt)


# -----------------------------
# SIR simulation function
# -----------------------------
def simulate_sir(transmission_rate, recovery_days, immunization_speed):
    recovery_rate = 1 / recovery_days

    def sir_system(t, state):
        S, I, R, V = state

        new_infections = transmission_rate * S * I
        new_recoveries = recovery_rate * I
        new_vaccinations = min(immunization_speed, S) if S > 0 else 0

        dS_dt = -new_infections - new_vaccinations
        dI_dt = new_infections - new_recoveries
        dR_dt = new_recoveries
        dV_dt = new_vaccinations

        return [dS_dt, dI_dt, dR_dt, dV_dt]

    solution = solve_ivp(
        sir_system,
        (time[0], time[-1]),
        [S0, I0, R0, V0],
        t_eval=time,
    )

    if not solution.success:
        raise RuntimeError(f"SIR integration failed: {solution.message}")

    return solution.y


# -----------------------------
# Widgets
# -----------------------------
beta_slider = widgets.FloatSlider(
    value=initial_transmission_rate,
    min=0.0001,
    max=0.01,
    step=0.0001,
    description="β:",
    readout_format=".4f"
)

recovery_slider = widgets.IntSlider(
    value=initial_recovery_days,
    min=1,
    max=30,
    step=1,
    description="Recovery days:"
)

vaccinated_slider = widgets.FloatSlider(
    value=initial_immunization_speed,
    min=0.0,
    max=200,
    step=0.0001,
    description="Vaccination speed:",
)

reset_button = widgets.Button(
    description="Reset",
    button_style="warning"
)

output = widgets.Output()


# -----------------------------
# Plot function
# -----------------------------
def draw_plot():
    transmission_rate = beta_slider.value
    recovery_days = recovery_slider.value
    immunization_speed = vaccinated_slider.value

    S, I, R, V = simulate_sir(transmission_rate, recovery_days, immunization_speed)

    with output:
        clear_output(wait=True)

        plt.figure(figsize=(10, 6))
        plt.plot(time, S, label="Susceptible S(t)")
        plt.plot(time, I, label="Infectious I(t)")
        plt.plot(time, R, label="Recovered R(t)")
        plt.plot(time, V, label="Vaccinated V(t)")

        plt.title(
            f"SIR Model: β = {transmission_rate:.4f}, "
            f"recovery time = {recovery_days} days"
        )
        plt.xlabel("Time in days")
        plt.ylabel("Number of people")
        plt.ylim(0, max(S.max(), I.max(), R.max(), V.max()) * 1.1)
        plt.legend()
        plt.grid(True)
        plt.show()


# -----------------------------
# Update function
# -----------------------------
def update_plot(change=None):
    draw_plot()


# -----------------------------
# Reset function
# -----------------------------
def reset_model(button):
    beta_slider.value = initial_transmission_rate
    recovery_slider.value = initial_recovery_days
    vaccinated_slider.value = initial_immunization_speed
    draw_plot()


beta_slider.observe(update_plot, names="value")
recovery_slider.observe(update_plot, names="value")
vaccinated_slider.observe(update_plot, names="value")
reset_button.on_click(reset_model)


display(beta_slider, recovery_slider, vaccinated_slider, reset_button, output)

draw_plot()

FloatSlider(value=0.0025, description='β:', max=0.01, min=0.0001, readout_format='.4f', step=0.0001)

IntSlider(value=7, description='Recovery days:', max=30, min=1)

FloatSlider(value=0.01, description='Vaccination speed:', max=200.0, step=0.0001)

Button(button_style='warning', description='Reset', style=ButtonStyle())

Output()